<a href="https://colab.research.google.com/github/AnitaPannerselvam/cdwmain/blob/day5/Day_5_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from transformers import GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, GPT2LMHeadModel, pipeline, \
                         Trainer, TrainingArguments

In [4]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')  # load up a standard gpt2 model

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [5]:
# load up our data into a dataset
pds_data = TextDataset(
    tokenizer=tokenizer,
    file_path='/content/Book.txt',  # Principles of Data Science - Sinan Ozdemir
    block_size=64  # length of each chunk of text to use as a datapoint
)

/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [6]:
pds_data[0], pds_data[0].shape  # inspect the first point

(tensor([   44,  7156,  1404, 25994,   198,    44,  7156,  1404, 25994,   198,
            51, 25994,    45, 43781,  3268,   362,    46,    20,    46,   198,
            51, 25994,    45, 43781,  3268, 32215,   198, 42131,   416,   198,
            35, 43664,  3698,  8782, 15154, 34509, 42131,   416,   198,    35,
         43664,  3698,  8782, 15154, 34509,   198, 30650,   198,   198, 24492,
           739,  8568, 17098,   422,   383, 36619,   416,   198, 37046, 13661,
         12052,   198,    18,  6479]),
 torch.Size([64]))

In [7]:
print(tokenizer.decode(pds_data[0]))

MEGATECH
MEGATECH
TECHNOLOGY IN 2O5O
TECHNOLOGY IN 2050
edited by
DANIEL FRANKLINedited by
DANIEL FRANKLIN
Books

Published under exclusive licence from The Economist by
Profile Books Ltd
3 Hol


In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
    # MLM is Masked Language Modelling (for BERT + auto-encoding tasks)
)

In [9]:
# example of how collator pads data dynamically
collator_example = data_collator([tokenizer('I am an input'), tokenizer('So am I')])

collator_example

{'input_ids': tensor([[   40,   716,   281,  5128],
        [ 2396,   716,   314, 50256]]), 'attention_mask': tensor([[1, 1, 1, 1],
        [1, 1, 1, 0]]), 'labels': tensor([[  40,  716,  281, 5128],
        [2396,  716,  314, -100]])}

In [10]:
collator_example.input_ids  # 50256 is our pad token id

tensor([[   40,   716,   281,  5128],
        [ 2396,   716,   314, 50256]])

In [11]:
tokenizer.pad_token_id

50256

In [12]:
collator_example.attention_mask  # Note the 0 in the attention mask where we have a pad token

tensor([[1, 1, 1, 1],
        [1, 1, 1, 0]])

In [13]:
collator_example.labels  # note the -100 to ignore loss calculation for the padded token
# Labels are shifted inside the GPT model so we don't need to worry about that

tensor([[  40,  716,  281, 5128],
        [2396,  716,  314, -100]])

In [14]:
model = GPT2LMHeadModel.from_pretrained('gpt2')  # load up a GPT2 model

pretrained_generator = pipeline(  # create a generator with built in params
    'text-generation', model=model, tokenizer='gpt2',
    config={'max_length': 200, 'do_sample': True, 'top_p': 0.9, 'temperature': 0.7, 'top_k': 10}
)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


In [15]:
print('----------')
for generated_sequence in pretrained_generator('This dataset shows the relationship', num_return_sequences=3):
    print(generated_sequence['generated_text'])
    print('----------')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------
This dataset shows the relationship between family size, marital status (couple size) and parental income, as well as the relationship between maternal and paternal wealth. However, the relationships between paternal wealth and paternal age were not clearly defined.

Figure 1
----------
This dataset shows the relationship between obesity incidence and prevalence of metabolic diseases, obesity incidence and age at first diagnosis (CHD). Data from this dataset are based on the NICE-funded National Longitudinal Study of Adolescent Health, which included more than
----------
This dataset shows the relationship between the number and quality of sleep, while also showing the role of sleep restriction on the proportion of non-active (nonreserved) sleep. When the study had only 2 months of sleep, people who abstained from
----------


In [16]:
training_args = TrainingArguments(
    output_dir="./gpt2_pds", #The output directory
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=3, # number of training epochs
    per_device_train_batch_size=32, # batch size for training
    per_device_eval_batch_size=32,  # batch size for evaluation
    logging_steps=10,
    load_best_model_at_end=True,
    evaluation_strategy='epoch',
    save_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=pds_data.examples[:int(len(pds_data.examples)*.8)],
    eval_dataset=pds_data.examples[int(len(pds_data.examples)*.8):]
)

trainer.evaluate()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'eval_loss': 4.3666510581970215,
 'eval_model_preparation_time': 0.0037,
 'eval_runtime': 0.2441,
 'eval_samples_per_second': 102.429,
 'eval_steps_per_second': 4.097}

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,3.863724,0.003700
2,No log,3.801314,0.003700
3,4.281300,3.781181,0.003700


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=12, training_loss=4.22056766351064, metrics={'train_runtime': 86.1261, 'train_samples_per_second': 3.379, 'train_steps_per_second': 0.139, 'total_flos': 9504497664000.0, 'train_loss': 4.22056766351064, 'epoch': 3.0})

In [18]:
trainer.evaluate()  # loss decrease is slowing down so we are hitting our limit

{'eval_loss': 3.7811810970306396,
 'eval_model_preparation_time': 0.0037,
 'eval_runtime': 0.1628,
 'eval_samples_per_second': 153.517,
 'eval_steps_per_second': 6.141,
 'epoch': 3.0}

In [19]:
trainer.save_model()

In [21]:
loaded_model = GPT2LMHeadModel.from_pretrained('./gpt2_pds')

finetuned_generator = pipeline(
    'text-generation', model=loaded_model, tokenizer=tokenizer,
    config={'max_length': 200, 'do_sample': True, 'top_p': 0.9, 'temperature': 0.7, 'top_k': 10}
)

Device set to use cuda:0


In [23]:
# examples are now sustainably about data
print('----------')
for generated_sequence in finetuned_generator('what is megatech', num_return_sequences=3):
    print(generated_sequence['generated_text'])
    print('----------')

----------
what is megatech-nazi and what does it mean?). Is this just a figment of my imagination?"


If Trump is truly an unhinged and unpredictable brand of personality and character, then surely the way he plays it out
----------
what is megatechia] is at the mouth of the global-economic-complex." (Rigotti 1992, p. 9)
It is quite clear that such things are at play. A recent book, The Great Transformation: The
----------
what is megatechooge?)
This post may contain links to Amazon or other partners; your purchases via these links can benefit Serious Eats. Read more about our affiliate linking policy.
----------
